In [1]:
import numpy as np
import pickle
from dataclasses import dataclass
from PIL import Image
import os

dir_path = "data/recorded_trajectory/mpc/debugged/"
subdir_names = ["cube_combi",
                "cube_roll",
                "cuboid_combi",
                "cuboid_pitch",
                "cuboid_translate",
                "cylinder_pitch",
                "cylinder_translate",
                "cube_pitch",
                "cube_translate",
                # "cuboid_combi_1",
                "cuboid_roll",
                "cylinder_combi",
                "cylinder_roll"]

raw_data = {}
for subdir_name in subdir_names:
    file_dir = dir_path + subdir_name 
    # find the total number of files in file_dir
    file_list = [f for f in os.listdir(file_dir) if f.endswith('.npz')]
    imgs = []
    final_target_poses = []
    target_poses = []
    real_poses = []
    for file_idx in range(len(file_list)):
        file_path = os.path.join(file_dir, f"{file_idx}.npz")
        print(f"Processing {file_path}...")

        # Load npz file
        data = np.load(file_path)
        final_target_pose = data['final_target_pose']
        target_pose = data['target_pose']
        real_pose = data['real_pose']
        img = data['img']
        imgs.append(Image.fromarray(img))
        final_target_poses.append(final_target_pose)
        target_poses.append(target_pose)
        real_poses.append(real_pose)
    
    raw_data[subdir_name] = {
        'imgs': imgs,
        'final_target_poses': final_target_poses,
        'target_poses': target_poses,
        'real_poses': real_poses
    }

    # Save as GIF
    # imgs[0].save(dir_path+f'{subdir_name}.gif', save_all=True, append_images=imgs[1:], duration=1000/5, loop=0)

In [2]:
import open3d as o3d

def visualize_mesh_at_start_end_target_poses(mesh_path, start_pose, end_pose, target_pose):
    # Create 3 mesh copies with different colors
    mesh_start = o3d.io.read_triangle_mesh(mesh_path)
    mesh_start.compute_vertex_normals()
    mesh_start.paint_uniform_color([0.5, 0.5, 0.5])  # gray
    mesh_start.transform(start_pose)

    mesh_end = o3d.io.read_triangle_mesh(mesh_path)
    mesh_end.compute_vertex_normals()
    mesh_end.paint_uniform_color([0.0, 0.0, 1.0])  # blue
    mesh_end.transform(end_pose)

    mesh_target = o3d.io.read_triangle_mesh(mesh_path)
    mesh_target.compute_vertex_normals()
    mesh_target.paint_uniform_color([0.0, 1.0, 0.0])  # green
    mesh_target.transform(target_pose)

    # Combine all meshes
    geometries = [mesh_start, mesh_end, mesh_target]

    # Set up headless rendering
    width, height = 800, 600
    renderer = o3d.visualization.rendering.OffscreenRenderer(width, height)
    # scene = renderer.scene

    # for geom in geometries:
    #     scene.add_geometry(str(id(geom)), geom, o3d.visualization.rendering.MaterialRecord())

    # # Set a good camera viewpoint
    # bbox = o3d.geometry.TriangleMesh()
    # for geom in geometries:
    #     bbox += geom
    # bounds = bbox.get_axis_aligned_bounding_box()
    # center = bounds.get_center()
    # extent = bounds.get_extent()
    # eye = center + [0, -2 * max(extent), max(extent)]
    # up = [0, 0, 1]
    # renderer.setup_camera(60.0, bounds, center)
    # renderer.scene.camera.look_at(center, eye, up)

    # # Render and save image
    # img = renderer.render_to_image()
    # o3d.io.write_image("rendered_image.png", img)
    # renderer.release()

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [3]:
import open3d as o3d
import numpy as np

def visualize_mesh_with_transparency_and_axes(mesh_path, start_pose, end_pose, target_pose, name=""):
    # 1. Load meshes and apply poses
    # ----------------------------------------
    mesh_start = o3d.io.read_triangle_mesh(mesh_path)
    mesh_start.compute_vertex_normals()
    mesh_start.transform(start_pose)

    mesh_end = o3d.io.read_triangle_mesh(mesh_path)
    mesh_end.compute_vertex_normals()
    mesh_end.transform(end_pose)

    mesh_target = o3d.io.read_triangle_mesh(mesh_path)
    mesh_target.compute_vertex_normals()
    mesh_target.transform(target_pose)

    # 2. Add coordinate frames for each pose (0.02m)
    # ----------------------------------------------
    axis_size = 0.02
    axis_start = o3d.geometry.TriangleMesh.create_coordinate_frame(size=axis_size).transform(start_pose)
    axis_end = o3d.geometry.TriangleMesh.create_coordinate_frame(size=axis_size).transform(end_pose)
    axis_target = o3d.geometry.TriangleMesh.create_coordinate_frame(size=axis_size).transform(target_pose)

    # 3. Define Materials with Transparency (Alpha = 0.3 for 70% transparent)
    # -----------------------------------------------------------------------
    alpha = 0.7

    colors = [(0.368, 0.507, 0.71), (0.881, 0.611, 0.142),
          (0.56, 0.692, 0.195), (0.923, 0.386, 0.209),
          (0.528, 0.471, 0.701), (0.772, 0.432, 0.102),
          (0.364, 0.619, 0.782), (0.572, 0.586, 0.) ] 
    
    mat_start = o3d.visualization.rendering.MaterialRecord()
    mat_start.shader = "defaultLitTransparency"
    mat_start.base_color = [0.5, 0.5, 0.5, 0.8]  # Gray

    mat_end = o3d.visualization.rendering.MaterialRecord()
    mat_end.shader = "defaultLitTransparency"
    mat_end.base_color = [0.0, 0.0, 1.0, 1.0]  # Blue

    mat_target = o3d.visualization.rendering.MaterialRecord()
    mat_target.shader = "defaultLitTransparency"
    mat_target.base_color = [0.0, 1.0, 0.0, 1.0]  # Green

    # Standard opaque material for the coordinate axes
    mat_axes = o3d.visualization.rendering.MaterialRecord()
    mat_axes.shader = "defaultLit"

    # 4. Set up Offscreen Renderer
    # ----------------------------
    width, height = 960, 960
    renderer = o3d.visualization.rendering.OffscreenRenderer(width, height)
    
    # Optional: Set background color to white for better visibility of transparent objects
    renderer.scene.set_background([1.0, 1.0, 1.0, 1.0])

    # Add geometries to the scene
    renderer.scene.add_geometry("mesh_start", mesh_start, mat_start)
    renderer.scene.add_geometry("mesh_end", mesh_end, mat_end)
    renderer.scene.add_geometry("mesh_target", mesh_target, mat_target)
    
    renderer.scene.add_geometry("axis_start", axis_start, mat_axes)
    renderer.scene.add_geometry("axis_end", axis_end, mat_axes)
    renderer.scene.add_geometry("axis_target", axis_target, mat_axes)

    # 5. Configure the Camera
    # -----------------------
    fov = 30.0                               # Field of View
    center = [0,0,-0.02]              # Looking at: Center of the target pose
    eye = [0.15, -0.15, -0.0]                  # Looking from: Camera position
    up = [0.0, 0.5, -0.5]                    # Up side: -z
    near_clip = 0.01

    renderer.setup_camera(fov, center, eye, up, near_clip)

    # 6. Render and Save
    # ------------------
    img = renderer.render_to_image()
    output_filename = f"{name}_transparent.png" if name else "transparent_render.png"
    
    # Write the image to disk
    o3d.io.write_image(output_filename, img)
    print(f"Render saved successfully to: {output_filename}")

In [6]:
from geomstats.geometry.special_euclidean import SpecialEuclidean

def get_distances(current_pose: np.ndarray, target_poses: np.ndarray) -> float:
    se3 = SpecialEuclidean(n=3, point_type='matrix')
    metric = se3.metric*1.0
    return metric.dist(current_pose, target_poses)

In [7]:
def get_distances(current_pose: np.ndarray, target_poses: np.ndarray) -> float:
    se3 = SpecialEuclidean(n=3, point_type='matrix')
    metric = se3.metric*1.0

    angle_current_pose = np.eye(4)
    angle_target_poses = np.eye(4)
    angle_current_pose[:3,:3] = current_pose[:3,:3]
    angle_target_poses[:3,:3] = target_poses[:3,:3]
    return metric.dist(current_pose, target_poses)/np.sqrt(2), np.linalg.norm(current_pose[:3,3] - target_poses[:3,3])

In [8]:
obj_names = ["cuboid","cube","cylinder"]
combi_names = ["_translate","_combi","_roll", "_pitch"]
# mat = np.zeros([3,4,3],dtype=float)

num_objs = len(obj_names)
num_motions = len(combi_names)
mat = np.zeros([num_objs, num_motions, 3], dtype=float)
mat_angle = np.zeros([num_objs, num_motions, 3], dtype=float)
initial_mat = np.zeros([num_objs, num_motions, 3], dtype=float)
initial_angle_mat = np.zeros([num_objs, num_motions, 3], dtype=float)
    
for i in range(num_objs):
    for j in range(num_motions):
        for k in range(3):
            dist_angle, dist = get_distances(current_pose=raw_data[f"{obj_names[i]}{combi_names[j]}"]["real_poses"][-k], target_poses=raw_data[f"{obj_names[i]}{combi_names[j]}"]["final_target_poses"][0])
            mat_angle[i][j][k], mat[i][j][k] = dist_angle, dist

In [ ]:
obj_names = ["cuboid","cube","cylinder"]
combi_names = ["_translate","_combi","_roll", "_pitch"]
mat = np.zeros([3,4,3],dtype=float)
    
for i in range(3):
    for j in range(4):
        for k in range(3):
            mat[i][j][k] = get_distances(current_pose=raw_data[f"{obj_names[i]}{combi_names[j]}"]["real_poses"][-k], target_poses=raw_data[f"{obj_names[i]}{combi_names[j]}"]["final_target_poses"][0])

In [30]:
mat[:,:,-1].mean(axis=0),mat[:,:,-1].std(axis=0) # translation error

(array([0.00485752, 0.00807045, 0.0067183 , 0.01091942]),
 array([0.00156249, 0.003753  , 0.00407195, 0.00455839]))

In [28]:
np.rad2deg(mat_angle[:,:,-1].mean(axis=0)),np.rad2deg(mat_angle[:,:,-1].std(axis=0)) # rotation error


(array([ 9.10571588, 19.89772806, 12.10489652, 42.07906546]),
 array([ 3.32213155, 10.49808454,  7.29437874,  6.98636657]))

In [17]:
mat_angle.min(axis=-1)

array([[0.03595508, 0.22210808, 0.20861465, 0.60898123],
       [0.01519649, 0.10744223, 0.19967215, 0.89961408],
       [0.03626144, 0.55214604, 0.03150267, 0.65815596]])

In [5]:
mat_a = np.min(mat,axis=-1)
mat_mean = np.mean(mat_a, axis=0)
mat_var = np.var(mat_a, axis=0)

In [6]:
mat_a

array([[0.05084817, 0.31410827, 0.29502567, 0.86122951],
       [0.02149109, 0.15194626, 0.28237907, 1.27224643],
       [0.05128141, 0.78085242, 0.0445515 , 0.93077308]])

In [7]:
print(f"mean: {mat_mean}, var: {mat_var**0.5}")

mean: [0.04120689 0.41563565 0.20731874 1.02141634], var: [0.0139423  0.26659782 0.11520957 0.1796216 ]


In [8]:
mesh_dir = "data/benchmark_objs/"

visualize_mesh_at_start_end_target_poses(
    mesh_path=mesh_dir+"cuboid_15x25mm.stl",
    start_pose=raw_data["cuboid_combi_1"]["real_poses"][0],
    end_pose=raw_data["cuboid_combi_1"]["real_poses"][-1],
    target_pose=raw_data["cuboid_combi_1"]["final_target_poses"][0]
)